[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/padillacm/IA-practice/blob/master/tokenization-embeddings-course/07_hnet_tokenizer_transfer/07_proyecto_hnet_tokenizer_transfer.ipynb)

# Proyecto 07 · Cambia el tokenizer de un modelo pequeño y mide qué pierdes (y qué ganas)

| | |
|---|---|
| **Nivel** | 🔴 Experto |
| **Duración estimada** | ≈ 2.5 h |
| **GPU recomendada** | `FAST_DEV_RUN=True`: CPU (≈ 5-8 min). Modo completo: **L4** (≈ 6-10 unidades de Colab, estimación) con **GPT-2 small** real (requiere acceso al hub de Hugging Face) |
| **Prerrequisitos** | Lección 07 (tokenizer transfer, FVT, FOCUS, bpb), módulos 02-04 |

## Contexto de negocio

Eres ML engineer en **Políglota Labs**. El equipo tiene un modelo pequeño preentrenado con **inglés y código** y un tokenizer BPE entrenado en eso: en español produce ~2x más tokens por byte
(más latencia y coste por consulta). El CTO pregunta: *«¿podemos cambiarle el tokenizer por uno entrenado en español sin reentrenar de cero, y cuánto calidad perdemos?»*
Tu trabajo: implementar **tres inicializaciones** del vocabulario nuevo (media, FVT, FOCUS-lite), hacer un **ajuste corto** y **medir bits por byte antes y después**, con un **control de regresión**
en inglés/código.

**Conexión.** Módulo anterior (06) y la primera mitad de la lección 07: ¿tokenizer-free o transfer? Aquí haces la opción pragmática. Módulo siguiente (09): la capa de embedding y la extensión de vocabulario con tokens de dominio — las mismas ideas aplicadas a un encoder de Políglota.

## Dataset
Corpus Políglota vía `carga_corpus()` (módulo 00). *Origen:* inglés + código; *destino:* español. Si imprime «SINTÉTICO», los valores absolutos cambian pero el protocolo es el mismo.

## Entregables y criterios de evaluación

1. `init_media`, `init_fvt`, `init_focus` (TODO 1-3) que pasen las comprobaciones unitarias.
2. `adapta` (TODO 4): ajuste en dos fases (solo embeddings → todo el modelo).
3. Tabla/curvas de bpb en español: **cero-shot**, **tras la fase de embeddings** y **final**, para cada init y para una init aleatoria.
4. **Regresión** en inglés/código tras adaptar al español.
5. Una **conclusión** en `CONCLUSION` (≥ 300 caracteres) con recomendación y límites.

| Criterio | Umbral (calibrado con la solución de referencia) |
|---|---|
| R1 · comprobaciones unitarias de las 3 inits | todas ✅ |
| R2 · fertilidad: bytes/token de B ÷ bytes/token de A en ES | ≥ 1.6x (medido: 1.90x) |
| R3 · bpb tras la fase de solo embeddings (K1 pasos), FVT vs aleatoria | FVT ≤ aleatoria − 0.03 (medido: 0.703 vs 0.779 con FAST_DEV_RUN; en cero-shot la diferencia es mínima, ≈ 0.01 bpb, y no se puntúa) |
| R4 · bpb final en ES con la mejor init | ≤ 0.3 (medido: 0.232, FVT) |
| R5 · adaptado vs modelo base sin adaptar (tokenizer A) | menos bpb |
| R6 · regresión en EN+código (aumento relativo de bpb) | ≤ 60 % (medido: +50 %: el ajuste al ES **sí** hace olvidar) |
| R7 · conclusión | ≥ 300 caracteres |

> Los umbrales salen de lo **medido** por la solución (FAST_DEV, corpus sintético, una semilla); con datos reales o GPT-2 real, vuelve a medir y ajusta.

In [ ]:
!pip install -q torch numpy matplotlib tokenizers transformers datasets scipy

In [ ]:
import os, sys, time, math, random, warnings
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
warnings.filterwarnings("ignore")
import numpy as np
import torch

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
FAST_DEV_RUN = True      # True: CPU, minutos. False: GPU (T4/L4), escala completa
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    torch.set_num_threads(int(os.environ.get("OMP_NUM_THREADS", min(4, os.cpu_count() or 1))))
print("device:", device, "| FAST_DEV_RUN:", FAST_DEV_RUN, "| torch", torch.__version__)

import re
import matplotlib.pyplot as plt
import torch.nn as nn, torch.nn.functional as F
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False})
AZUL, NARANJA, VERDE, GRIS = "#1F6FEB", "#F28E2B", "#2CA02C", "#6B7280"

import copy, re
from collections import Counter

In [ ]:
# ── Datos para el corpus SINTÉTICO de Políglota (último recurso si no hay red / Hugging Face) ──
# (artículo+concepto ES, artículo+concepto EN, definición ES, definición EN)
CONCEPTOS = [
    ("el índice invertido", "the inverted index", "una estructura que asocia cada término con la lista de documentos que lo contienen", "a structure that maps each term to the list of documents that contain it"),
    ("la caché", "the cache", "una memoria rápida que guarda resultados recientes para no recalcularlos", "a fast memory that stores recent results so they are not recomputed"),
    ("la base de datos relacional", "the relational database", "un sistema que organiza los datos en tablas relacionadas mediante claves", "a system that organizes data into tables related through keys"),
    ("el balanceador de carga", "the load balancer", "un componente que reparte las peticiones entre varios servidores", "a component that spreads requests across several servers"),
    ("el contenedor Docker", "the Docker container", "un proceso aislado que empaqueta una aplicación con sus dependencias", "an isolated process that packages an application with its dependencies"),
    ("el control de versiones", "version control", "un sistema que registra los cambios de los archivos a lo largo del tiempo", "a system that records changes to files over time"),
    ("la API REST", "the REST API", "una interfaz que expone recursos mediante URL y verbos HTTP", "an interface that exposes resources through URLs and HTTP verbs"),
    ("la autenticación con tokens", "token authentication", "un mecanismo que identifica al usuario con un testigo firmado en cada petición", "a mechanism that identifies the user with a signed token on every request"),
    ("la expresión regular", "the regular expression", "un patrón que describe un conjunto de cadenas de texto", "a pattern that describes a set of text strings"),
    ("el hilo de ejecución", "the thread", "una secuencia de instrucciones que comparte memoria con otros hilos del proceso", "a sequence of instructions that shares memory with other threads of the process"),
    ("el recolector de basura", "the garbage collector", "un mecanismo que libera automáticamente la memoria que ya no se usa", "a mechanism that automatically frees memory that is no longer used"),
    ("la cola de mensajes", "the message queue", "un búfer que desacopla a los productores de los consumidores de eventos", "a buffer that decouples event producers from consumers"),
    ("la tabla hash", "the hash table", "una estructura que guarda pares clave-valor y busca en tiempo casi constante", "a structure that stores key-value pairs and looks them up in near-constant time"),
    ("el árbol binario de búsqueda", "the binary search tree", "un árbol ordenado donde cada nodo separa los valores menores de los mayores", "an ordered tree where each node separates smaller values from larger ones"),
    ("la red neuronal", "the neural network", "un modelo formado por capas de unidades que aprenden pesos a partir de ejemplos", "a model made of layers of units that learn weights from examples"),
    ("el descenso de gradiente", "gradient descent", "un método que ajusta los parámetros en la dirección que reduce la pérdida", "a method that adjusts parameters in the direction that reduces the loss"),
    ("el sobreajuste", "overfitting", "el fenómeno por el que un modelo memoriza el entrenamiento y generaliza mal", "the phenomenon where a model memorizes the training set and generalizes poorly"),
    ("la validación cruzada", "cross-validation", "una técnica que estima el rendimiento repartiendo los datos en varios pliegues", "a technique that estimates performance by splitting the data into several folds"),
    ("el tokenizador", "the tokenizer", "el componente que convierte el texto en una secuencia de identificadores enteros", "the component that turns text into a sequence of integer identifiers"),
    ("la ventana de contexto", "the context window", "la cantidad máxima de tokens que un modelo puede leer a la vez", "the maximum number of tokens a model can read at once"),
    ("el compilador", "the compiler", "un programa que traduce código fuente a instrucciones de máquina", "a program that translates source code into machine instructions"),
    ("el sistema de archivos", "the file system", "la capa que organiza cómo se guardan y se nombran los datos en disco", "the layer that organizes how data is stored and named on disk"),
    ("la memoria virtual", "virtual memory", "una abstracción que da a cada proceso su propio espacio de direcciones", "an abstraction that gives each process its own address space"),
    ("el protocolo TCP", "the TCP protocol", "un protocolo que entrega bytes de forma ordenada y fiable entre dos máquinas", "a protocol that delivers bytes reliably and in order between two machines"),
    ("el cifrado simétrico", "symmetric encryption", "un esquema que usa la misma clave para cifrar y descifrar", "a scheme that uses the same key to encrypt and decrypt"),
    ("las pruebas unitarias", "unit tests", "comprobaciones automáticas que verifican una función aislada", "automatic checks that verify a single isolated function"),
    ("la integración continua", "continuous integration", "una práctica que compila y prueba cada cambio de forma automática", "a practice that builds and tests every change automatically"),
    ("los microservicios", "microservices", "una arquitectura que divide la aplicación en servicios pequeños e independientes", "an architecture that splits the application into small independent services"),
    ("la paginación", "pagination", "una técnica que devuelve los resultados en bloques de tamaño fijo", "a technique that returns results in fixed-size blocks"),
    ("la limitación de tasa", "rate limiting", "una política que restringe cuántas peticiones puede hacer un cliente por minuto", "a policy that restricts how many requests a client can make per minute"),
    ("la serialización JSON", "JSON serialization", "el proceso de convertir objetos en texto JSON y viceversa", "the process of converting objects to JSON text and back"),
    ("la variable de entorno", "the environment variable", "un par nombre-valor que el sistema operativo entrega a cada proceso", "a name-value pair that the operating system hands to each process"),
    ("el entorno virtual", "the virtual environment", "una carpeta aislada con su propio intérprete de Python y sus paquetes", "an isolated folder with its own Python interpreter and packages"),
    ("el decorador", "the decorator", "una función que envuelve a otra para añadirle comportamiento", "a function that wraps another one to add behavior"),
    ("el generador", "the generator", "una función que produce valores perezosamente con la palabra yield", "a function that lazily produces values with the yield keyword"),
    ("el manejo de excepciones", "exception handling", "el mecanismo para capturar errores sin que el programa se detenga", "the mechanism for catching errors without stopping the program"),
    ("la programación asíncrona", "asynchronous programming", "un modelo que solapa esperas de entrada y salida sin bloquear el hilo", "a model that overlaps input and output waits without blocking the thread"),
    ("el bloqueo de base de datos", "the database lock", "un candado que impide que dos transacciones modifiquen la misma fila a la vez", "a lock that prevents two transactions from modifying the same row at once"),
    ("la migración de esquema", "the schema migration", "un script versionado que cambia la estructura de la base de datos", "a versioned script that changes the structure of the database"),
    ("el registro de eventos", "event logging", "la práctica de escribir mensajes con marca de tiempo sobre lo que hace el sistema", "the practice of writing timestamped messages about what the system does"),
]
HERRAMIENTAS = ["PostgreSQL", "Redis", "nginx", "Docker", "pytest", "Kubernetes", "FastAPI", "SQLite", "Git", "Linux", "Celery", "Prometheus"]
PARAMETROS = ["max_connections", "timeout_seconds", "cache_size", "batch_size", "log_level", "retry_count", "pool_size", "num_workers"]
# aspecto -> (título ES, título EN, palabra clave ES, palabra clave EN, plantillas ES, plantillas EN)
ASPECTOS = {
    "intro": ("{n}: qué es y para qué sirve", "{n}: what it is and what it is for", "qué es", "what is",
        ["Se usa en {t} para simplificar el trabajo diario del equipo.", "Entender {c} es un requisito antes de pasar a producción.", "Su idea central es sencilla y aparece en casi todos los sistemas modernos.", "La documentación de {t} dedica un capítulo completo a {c}."],
        ["It is used in {t} to simplify the daily work of the team.", "Understanding {c} is a prerequisite before going to production.", "Its core idea is simple and appears in almost every modern system.", "The {t} documentation devotes a whole chapter to {c}."]),
    "config": ("{n}: cómo configurarlo", "{n}: how to configure it", "configurar", "configure",
        ["Para configurar {c} edita el archivo de ajustes y reinicia {t}.", "El parámetro {p} controla el comportamiento de {c} y su valor por defecto es {x}.", "Después de cambiar {p} a {x}, comprueba el resultado con un comando de diagnóstico.", "Guarda la configuración de {c} en un archivo versionado y no en la consola."],
        ["To configure {c}, edit the settings file and restart {t}.", "The {p} parameter controls the behavior of {c} and its default value is {x}.", "After changing {p} to {x}, check the result with a diagnostic command.", "Keep the configuration of {c} in a versioned file and not in the console."]),
    "errores": ("{n}: errores frecuentes", "{n}: common errors", "errores", "errors",
        ["Un error frecuente con {c} es olvidar el valor de {p} y dejarlo en {x}.", "Si aparece un fallo intermitente en {t}, revisa primero los registros de {c}.", "Muchos equipos descubren el problema de {c} solo cuando la carga se multiplica por {x}.", "El mensaje de error suele ser engañoso: la causa real está en {c}, no en {t}."],
        ["A common mistake with {c} is forgetting {p} and leaving it at {x}.", "If an intermittent failure shows up in {t}, check the logs of {c} first.", "Many teams only discover the problem with {c} when the load grows by {x} times.", "The error message is usually misleading: the real cause is in {c}, not in {t}."]),
    "rendimiento": ("{n}: rendimiento y optimización", "{n}: performance and optimization", "rendimiento", "performance",
        ["Para medir el rendimiento de {c} usa un perfilador y no tu intuición.", "Subir {p} a {x} mejora la latencia, pero consume más memoria en {t}.", "La optimización prematura de {c} suele complicar el código sin ganar tiempo.", "En una prueba con {x} usuarios, {c} redujo la latencia mediana a la mitad."],
        ["To measure the performance of {c}, use a profiler and not your intuition.", "Raising {p} to {x} improves latency but uses more memory in {t}.", "Premature optimization of {c} usually complicates the code without saving time.", "In a test with {x} users, {c} cut the median latency in half."]),
    "practicas": ("{n}: buenas prácticas", "{n}: best practices", "buenas prácticas", "best practices",
        ["Una buena práctica con {c} es documentar {p} y automatizar su despliegue en {t}.", "Revisa {c} en cada revisión de código y escribe pruebas para los casos límite.", "Mantén {c} simple: añade complejidad solo cuando los datos lo justifiquen.", "Define alertas sobre {c} antes del primer incidente, no después."],
        ["A good practice with {c} is to document {p} and automate its deployment in {t}.", "Review {c} in every code review and write tests for the edge cases.", "Keep {c} simple: add complexity only when the data justifies it.", "Define alerts on {c} before the first incident, not after."]),
    "ejemplo": ("{n}: ejemplo en Python", "{n}: Python example", "ejemplo", "example",
        ["En este ejemplo en Python usamos {t} para mostrar {c} paso a paso.", "El script define {p} = {x} y luego imprime el resultado de {c}.", "Ejecuta el ejemplo con Python 3.11 y compara la salida con la esperada.", "Puedes adaptar el código de {c} a tu propio proyecto cambiando solo {p}."],
        ["In this Python example we use {t} to show {c} step by step.", "The script defines {p} = {x} and then prints the result of {c}.", "Run the example with Python 3.11 and compare the output with the expected one.", "You can adapt the code for {c} to your own project by changing only {p}."]),
    "seguridad": ("{n}: consideraciones de seguridad", "{n}: security considerations", "seguridad", "security",
        ["Desde el punto de vista de la seguridad, {c} nunca debe exponerse sin autenticación.", "Rota las credenciales de {t} cada {x} días y no las guardes en el repositorio.", "Un ataque contra {c} suele aprovechar el valor de {p} mal configurado.", "Audita los accesos a {c} y conserva los registros durante {x} días."],
        ["From a security standpoint, {c} must never be exposed without authentication.", "Rotate the {t} credentials every {x} days and do not store them in the repository.", "An attack on {c} usually exploits a misconfigured {p} value.", "Audit access to {c} and keep the logs for {x} days."]),
}

In [ ]:
# ── Generador del corpus sintético (mismo esquema que el real) ────────────────
import random


def _es_fix(s: str) -> str:
    """Contracciones del español: 'a el' -> 'al', 'de el' -> 'del'."""
    return s.replace(" a el ", " al ").replace(" de el ", " del ").replace("De el ", "Del ").replace("A el ", "Al ")


def _sintetico_wiki(lang: str, n_docs: int, n_queries: int, seed: int = 42):
    """Docs técnicos sintéticos ES/EN: concepto x aspecto x herramienta. Devuelve (docs, queries, qrels)."""
    rng = random.Random(seed + (0 if lang == "es" else 1))
    li = 0 if lang == "es" else 1
    combos = [(c, a, t) for c in range(len(CONCEPTOS)) for a in ASPECTOS for t in range(len(HERRAMIENTAS))]
    rng.shuffle(combos)
    docs, titles = [], []
    for k, (ci, asp, ti) in enumerate(combos[:n_docs]):
        concepto, definicion = CONCEPTOS[ci][li], CONCEPTOS[ci][2 + li]
        tool = HERRAMIENTAS[ti]
        tit_t, plantillas = ASPECTOS[asp][li], ASPECTOS[asp][4 + li]
        nombre = concepto.split(" ", 1)[1] if concepto.split(" ", 1)[0] in ("el", "la", "los", "las", "the") else concepto
        slots = lambda: dict(c=concepto, t=tool, p=rng.choice(PARAMETROS), x=rng.choice([8, 16, 30, 64, 100, 256, 1024]))
        frases = [("{C} " + ("es " if li == 0 else "is ") + definicion + ".").format(C=concepto)]
        frases += [pl.format(**slots()) for pl in rng.sample(plantillas, len(plantillas))]
        frases.append(rng.choice(ASPECTOS["intro"][4 + li]).format(**slots()))
        for _ in range(2):  # ruido: frases de otro concepto (hace el retrieval no trivial)
            otro = rng.randrange(len(CONCEPTOS))
            otra = rng.choice(list(ASPECTOS.values()))[4 + li]
            s = slots(); s["c"] = CONCEPTOS[otro][li]
            frases.append(rng.choice(otra).format(**s))
        s = slots()
        frases.append(("Ejemplo: `{p} = {x}`." if li == 0 else "Example: `{p} = {x}`.").format(**s))
        text = " ".join(f[0].upper() + f[1:] for f in frases)
        docs.append({"id": f"{lang}-syn-{k:04d}", "lang": lang, "text": _es_fix(text) if lang == "es" else text})
        titles.append((nombre, ASPECTOS[asp][2 + li], tool, tit_t.format(n=nombre) + f" ({tool})"))
    queries, qrels = [], {}
    for j, k in enumerate(rng.sample(range(len(docs)), min(n_queries, len(docs)))):
        nombre, kw, tool, titulo = titles[k]
        q = titulo.replace(":", "").replace("(", "").replace(")", "") if j % 2 == 0 else f"{nombre} {kw} {tool}"
        qid = f"{lang}-q{j:04d}"
        queries.append({"qid": qid, "lang": lang, "text": q})
        qrels[qid] = {docs[k]["id"]: 1}
    return docs, queries, qrels

In [ ]:
# ── carga_corpus(): cadena oficial (HF) -> espejo (API de Wikipedia / stdlib) -> sintético ─────
import ast
import json
import os
import random
import urllib.parse
import urllib.request

SEED = 42


def _strip_docstring(func_src: str):
    """Devuelve (código sin docstring, primera frase del docstring) o None si no hay docstring útil."""
    try:
        fn = ast.parse(func_src).body[0]
        doc = ast.get_docstring(fn)
        first = fn.body[0]
    except Exception:
        return None
    if not doc or not (isinstance(first, ast.Expr) and isinstance(getattr(first, "value", None), ast.Constant)):
        return None
    lines = func_src.splitlines()
    code = "\n".join(lines[: first.lineno - 1] + lines[first.end_lineno :])
    frase = doc.strip().split("\n\n")[0].replace("\n", " ").strip()
    frase = frase.split(". ")[0].rstrip(".")
    return (code, frase) if len(frase.split()) >= 4 else None


def _code_stdlib(n_docs: int, n_queries: int, rng: random.Random):
    """Espejo OFFLINE con código real: funciones de la biblioteca estándar con su docstring como query."""
    import importlib, inspect, textwrap
    mods = ["argparse", "bisect", "calendar", "collections", "colorsys", "copy", "csv", "difflib", "fnmatch", "fractions",
            "functools", "getpass", "glob", "heapq", "html", "json", "operator", "os", "pathlib", "posixpath", "queue",
            "random", "re", "shlex", "shutil", "statistics", "string", "tempfile", "textwrap", "urllib.parse", "uuid", "zipfile"]
    cands = []
    for name in mods:
        try:
            src = inspect.getsource(importlib.import_module(name))
            tree = ast.parse(src)
        except Exception:
            continue
        for node in ast.walk(tree):
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.end_lineno - node.lineno >= 4:
                seg = textwrap.dedent("\n".join(src.splitlines()[node.lineno - 1 : node.end_lineno]))
                r = _strip_docstring(seg)
                if r and 200 <= len(r[0]) <= 2500:
                    cands.append((f"{name}.{node.name}", r[0], r[1]))
    rng.shuffle(cands)
    return cands[:n_docs]


def _code_hf(n_docs: int, rng: random.Random):
    """Oficial: CodeSearchNet (Python) en streaming. Prueba varios repos por cambios de formato en `datasets`."""
    from datasets import load_dataset
    errs = []
    for repo, kw in [("code-search-net/code_search_net", {"name": "python"}), ("code_search_net", {"name": "python"}),
                     ("claudios/code_search_net", {"name": "python"}), ("Nan-Do/code-search-net-python", {})]:
        try:
            ds = load_dataset(repo, split="train", streaming=True, **kw).shuffle(seed=SEED, buffer_size=2000)
            out = []
            for ex in ds:
                src = ex.get("func_code_string") or ex.get("whole_func_string") or ex.get("code") or ""
                r = _strip_docstring(src) if '"""' in src or "'''" in src else None
                doc = (ex.get("func_documentation_string") or ex.get("docstring") or "").strip().split("\n\n")[0]
                doc = doc.replace("\n", " ").split(". ")[0].rstrip(".")
                code = r[0] if r else src
                if 200 <= len(code) <= 2500 and len(doc.split()) >= 4:
                    out.append((ex.get("func_name") or f"csn.{len(out)}", code, doc))
                if len(out) >= n_docs:
                    return out
        except Exception as e:  # noqa: BLE001
            errs.append(f"{repo}: {type(e).__name__}")
    raise RuntimeError("CodeSearchNet no disponible -> " + "; ".join(errs))


def _wiki_hf(lang: str, n_docs: int, cap: int):
    """Oficial: wikimedia/wikipedia (snapshot 20231101) en streaming con un buffer pequeño."""
    from datasets import load_dataset
    ds = load_dataset("wikimedia/wikipedia", f"20231101.{lang}", split="train", streaming=True)
    ds = ds.shuffle(seed=SEED, buffer_size=2000)
    out = []
    for ex in ds:
        if len(ex["text"]) >= 600 and len(ex["title"]) >= 3:
            out.append((f"{lang}-{ex['id']}", ex["title"], ex["text"][:cap]))
        if len(out) >= n_docs:
            break
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out


def _wiki_api(lang: str, n_docs: int):
    """Espejo: API pública de Wikipedia (artículos aleatorios, solo la sección introductoria; ~20 por petición)."""
    out, seen, tries = [], set(), 0
    while len(out) < n_docs and tries < 80:
        tries += 1
        qs = urllib.parse.urlencode({"action": "query", "format": "json", "generator": "random", "grnnamespace": 0,
                                     "grnlimit": 20, "prop": "extracts", "exintro": 1, "explaintext": 1, "exlimit": 20})
        req = urllib.request.Request(f"https://{lang}.wikipedia.org/w/api.php?{qs}",
                                     headers={"User-Agent": "poliglota-course/1.0 (curso educativo)"})
        with urllib.request.urlopen(req, timeout=20) as r:
            pages = json.load(r).get("query", {}).get("pages", {})
        for p in pages.values():
            if p["pageid"] not in seen and len(p.get("extract", "")) >= 300:
                seen.add(p["pageid"])
                out.append((f"{lang}-{p['pageid']}", p["title"], p["extract"]))
    if len(out) < n_docs // 2:
        raise RuntimeError("pocos artículos")
    return out[:n_docs]


def carga_corpus(fast_dev: bool = True) -> dict:
    """Corpus Políglota -> {"docs": [{id, lang, text}], "queries": [{qid, lang, text}], "qrels": {qid: {doc_id: 1}},
    "source": "real" | "synthetic"}. Cadena: oficial (HF) -> espejo (API Wikipedia; código de la stdlib) -> sintético."""
    rng = random.Random(SEED)
    n_wiki, n_code, q_wiki, q_code, cap = (150, 100, 40, 30, 1500) if fast_dev else (3000, 1500, 200, 150, 6000)
    docs, queries, qrels, source = [], [], {}, "real"
    wiki = {}
    for lang in ("es", "en"):
        for nombre, fn in (("HF wikimedia/wikipedia", lambda: _wiki_hf(lang, n_wiki, cap)),
                           ("API de Wikipedia (espejo)", lambda: _wiki_api(lang, min(n_wiki, 400)))):
            try:
                wiki[lang] = fn()
                print(f"[carga_corpus] Wikipedia {lang}: {len(wiki[lang])} artículos desde {nombre}")
                break
            except Exception as e:  # noqa: BLE001
                print(f"[carga_corpus] {nombre} ({lang}) falló: {type(e).__name__}: {str(e)[:80]}")
    if len(wiki) == 2:
        for lang, arts in wiki.items():
            docs += [{"id": i, "lang": lang, "text": t} for i, _, t in arts]
            for j, k in enumerate(rng.sample(range(len(arts)), min(q_wiki, len(arts)))):
                qid = f"{lang}-q{j:04d}"
                queries.append({"qid": qid, "lang": lang, "text": arts[k][1]})
                qrels[qid] = {arts[k][0]: 1}
    else:
        source = "synthetic"
        print("⚠️  [carga_corpus] SIN acceso a Wikipedia: uso texto ES/EN SINTÉTICO (mismo esquema). Los números NO son comparables con datos reales.")
        for lang in ("es", "en"):
            d, q, r = _sintetico_wiki(lang, n_wiki, q_wiki, SEED)
            docs += d; queries += q; qrels.update(r)
    try:
        code = _code_hf(n_code, rng)
        print(f"[carga_corpus] código: {len(code)} funciones desde CodeSearchNet (HF)")
    except Exception as e:  # noqa: BLE001
        print(f"[carga_corpus] CodeSearchNet falló ({str(e)[:90]}); uso funciones REALES de la stdlib de Python (espejo offline)")
        code = _code_stdlib(n_code, q_code, rng)
    for k, (name, src, doc) in enumerate(code):
        docs.append({"id": f"code-{k:04d}", "lang": "code", "text": src})
    for j, k in enumerate(rng.sample(range(len(code)), min(q_code, len(code)))):
        qid = f"code-q{j:04d}"
        queries.append({"qid": qid, "lang": "code", "text": code[k][2]})
        qrels[qid] = {f"code-{k:04d}": 1}
    return {"docs": docs, "queries": queries, "qrels": qrels, "source": source}

## Paso 0 · Datos, tokenizers y modelo base (dados)

Partimos el corpus (módulo 06), entrenamos el tokenizer A (inglés + código) y el B (español), y preentrenamos —o cargamos— el modelo base.

In [ ]:
corp = carga_corpus(FAST_DEV_RUN)
print("fuente del corpus:", corp["source"], "| docs:", len(corp["docs"]))
rng = random.Random(SEED)
docs = corp["docs"][:]; rng.shuffle(docs)
cut = int(0.9 * len(docs))
DOCS_TR, DOCS_VA = docs[:cut], docs[cut:]
MAX_CHARS = 1500 if FAST_DEV_RUN else 6000
LIMITE_BYTES = None if FAST_DEV_RUN else 3_000_000        # el modelo n-grama es Python puro: acota el modo completo

def texto(ds):
    return "\n\n".join(d["text"][:MAX_CHARS] for d in ds)

TXT_TR, TXT_VA = texto(DOCS_TR), texto(DOCS_VA)
TR = np.frombuffer(TXT_TR.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
VA = np.frombuffer(TXT_VA.encode("utf-8"), dtype=np.uint8).copy()[:LIMITE_BYTES]
print(f"bytes train={len(TR):,} | val={len(VA):,}")
for lang in ["es", "en", "code"]:
    t = "".join(d["text"][:MAX_CHARS] for d in DOCS_TR if d["lang"] == lang)
    print(f"{lang:>4}: {len(t):>8,} chars -> {len(t.encode()):>8,} bytes  ({len(t.encode())/max(1, len(t)):.3f} bytes/char)")

In [ ]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

def entrena_bpe(docs_, vocab_size):
    tk = Tokenizer(models.BPE())
    tk.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
    tk.decoder = decoders.ByteLevel()
    trn = trainers.BpeTrainer(vocab_size=vocab_size, initial_alphabet=pre_tokenizers.ByteLevel.alphabet(),
                              show_progress=False)
    tk.train_from_iterator((d["text"][:MAX_CHARS] for d in docs_), trn)
    return tk

def bytes_por_token(tk, textos):
    n_b = sum(len(t.encode()) for t in textos)
    return n_b / sum(len(tk.encode(t).ids) for t in textos)

In [ ]:
def bytes_to_unicode():
    """Mapa GPT-2 byte -> carácter imprimible (el que usa ByteLevel). 1 carácter == 1 byte en el vocabulario."""
    bs = list(range(33, 127)) + list(range(161, 173)) + list(range(174, 256)); cs = bs[:]; n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b); cs.append(256 + n); n += 1
    return dict(zip(bs, map(chr, cs)))
B2U = bytes_to_unicode(); U2B = {v: k for k, v in B2U.items()}
tok_a_bytes = lambda s: bytes(U2B[c] for c in s)

from tokenizers import Tokenizer
SRC = [d for d in DOCS_TR if d["lang"] in ("en", "code")]            # "idioma de origen" del modelo base
TGT = [d for d in DOCS_TR if d["lang"] == "es"]                       # idioma destino de la adaptación
TGT_VA = [d for d in DOCS_VA if d["lang"] == "es"]
V_A, V_B = (512, 768) if FAST_DEV_RUN else (50257, 24000)
TK_A = entrena_bpe(SRC, V_A)                                          # tokenizer original (inglés + código)
TK_B = entrena_bpe(TGT, V_B)                                          # tokenizer nuevo (español)
if not FAST_DEV_RUN:
    try:
        TK_A = Tokenizer.from_pretrained("gpt2")                     # modo completo: GPT-2 de verdad (requiere acceso a Hugging Face)
        V_A = TK_A.get_vocab_size()
    except Exception as e:
        print("(sin acceso al hub; sigo con el BPE entrenado en local)", type(e).__name__)

VA_ES = "\n\n".join(d["text"][:MAX_CHARS] for d in TGT_VA)
def bytes_x_token(tk, txt): return len(txt.encode()) / len(tk.encode(txt).ids)

In [ ]:
from transformers import GPT2Config, GPT2LMHeadModel
T_TOK = 64

def ids_stream(tk, docs_):
    return np.array(tk.encode("\n\n".join(d["text"][:MAX_CHARS] for d in docs_)).ids, dtype=np.int64)

def lote_ids(ids, B, T, g):
    ix = torch.randint(0, len(ids) - T - 1, (B,), generator=g)
    x = torch.stack([torch.from_numpy(ids[i:i + T]) for i in ix]); y = torch.stack([torch.from_numpy(ids[i + 1:i + T + 1]) for i in ix])
    return x.to(device), y.to(device)

def largo_bytes(tk):                                                   # bytes de cada token (en ByteLevel 1 carácter = 1 byte)
    return np.array([len(tok_a_bytes(t)) if all(c in U2B for c in t) else 1 for t, _ in sorted(tk.get_vocab().items(), key=lambda kv: kv[1])])

@torch.no_grad()
def bpb(model, tk, txt, T=T_TOK):
    """bits por byte sobre un texto (ventanas contiguas sin solape): la métrica comparable entre tokenizers."""
    ids = np.array(tk.encode(txt).ids, dtype=np.int64); LB = largo_bytes(tk); model.eval(); nll = nb = 0.0
    for i in range(0, len(ids) - T - 1, T):
        x = torch.from_numpy(ids[i:i + T])[None].to(device); y = torch.from_numpy(ids[i + 1:i + T + 1])[None].to(device)
        nll += F.cross_entropy(model(input_ids=x).logits[0], y[0], reduction="sum").item(); nb += LB[ids[i + 1:i + T + 1]].sum()
    return nll / math.log(2) / nb

def nuevo_gpt(vocab, d=128, L=3, H=4):
    return GPT2LMHeadModel(GPT2Config(vocab_size=vocab, n_positions=T_TOK, n_embd=d, n_layer=L, n_head=H,
                                      resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, bos_token_id=0, eos_token_id=0)).to(device)

def entrena_lm(model, ids, pasos, lr=2e-3, params=None, log_cada=None, evalfn=None, seed=SEED):
    opt = torch.optim.AdamW(params if params is not None else model.parameters(), lr=lr); g = torch.Generator().manual_seed(seed); curva = []
    for s in range(pasos + 1):
        if log_cada and s % log_cada == 0: curva.append((s, evalfn(model)))
        if s == pasos: break
        model.train(); x, y = lote_ids(ids, 16, T_TOK, g)
        loss = F.cross_entropy(model(input_ids=x).logits.reshape(-1, model.config.vocab_size), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
    return curva

# --- Modelo base: preentrenado SOLO con inglés + código y el tokenizer A ---
t0 = time.time(); torch.manual_seed(SEED)
IDS_A_SRC = ids_stream(TK_A, SRC)
if FAST_DEV_RUN or TK_A.get_vocab_size() != 50257:
    BASE = nuevo_gpt(TK_A.get_vocab_size()); entrena_lm(BASE, IDS_A_SRC, 500 if FAST_DEV_RUN else 3000)
else:
    BASE = GPT2LMHeadModel.from_pretrained("gpt2").to(device)         # modo completo: GPT-2 small preentrenado (124M)
VA_SRC = "\n\n".join(d["text"][:MAX_CHARS] for d in DOCS_VA if d["lang"] in ("en", "code"))
print(f"modelo base listo en {time.time()-t0:.0f}s | bpb(val EN+código, tokenizer A) = {bpb(BASE, TK_A, VA_SRC):.3f}")
print(f"bpb(val ES, tokenizer A) ANTES de adaptar = {bpb(BASE, TK_A, VA_ES):.3f}   <- el modelo no ha visto español")

Utilidades dadas: `sparsemax`, `descomposicion`, `base_copia` (copia los tokens compartidos y devuelve los índices nuevos), `aux_embeddings` (PPMI+SVD) e `instala_vocab` (sustituye la matriz de embeddings del modelo).

In [ ]:
def sparsemax(z):
    """Martins & Astudillo (2016): proyección euclídea al simplex; produce pesos EXACTAMENTE cero fuera del soporte."""
    zs, _ = torch.sort(z, descending=True); k = torch.arange(1, len(z) + 1, device=z.device)
    cs = zs.cumsum(0); sop = (1 + k * zs) > cs; kz = int(sop.sum()); tau = (cs[kz - 1] - 1) / kz
    return torch.clamp(z - tau, min=0)

def vocab_ordenado(tk): return [t for t, _ in sorted(tk.get_vocab().items(), key=lambda kv: kv[1])]

def descomposicion(tk_old, token_str):
    """Ids del tokenizer ANTIGUO que componen el texto de un token nuevo (FVT/ZeTT). Si no es UTF-8 válido, usa bytes sueltos."""
    try:
        txt = tok_a_bytes(token_str).decode("utf-8")
        ids = tk_old.encode(txt).ids
        if ids: return ids
    except Exception:
        pass
    v = tk_old.get_vocab(); return [v[c] for c in token_str if c in v]

def base_copia(E_A, tkA, tkB, std=0.02):
    """Matriz [V_B, d] con los tokens compartidos copiados (cadena idéntica) y el resto sin inicializar (ceros)."""
    vA, VB = tkA.get_vocab(), vocab_ordenado(tkB); E = torch.zeros(len(VB), E_A.shape[1]); nuevo = []
    for i, t in enumerate(VB):
        if t in vA: E[i] = E_A[vA[t]]
        else: nuevo.append(i)
    return E, nuevo



def aux_embeddings(ids, V, ventana=4, dim=32):
    """Embeddings estáticos baratos (PPMI + SVD) de tokens: el 'fastText de juguete' que usan WECHSEL/FOCUS como espacio auxiliar."""
    C = np.zeros((V, V), np.float32)
    for k in range(1, ventana + 1):
        np.add.at(C, (ids[:-k], ids[k:]), 1.0); np.add.at(C, (ids[k:], ids[:-k]), 1.0)
    pr = C.sum(1, keepdims=True) / C.sum(); pmi = np.log((C / C.sum() + 1e-12) / (pr @ pr.T + 1e-12)); ppmi = np.maximum(pmi, 0)
    U, S, _ = np.linalg.svd(ppmi, full_matrices=False); W = U[:, :dim] * np.sqrt(S[:dim])
    return torch.from_numpy(W / (np.linalg.norm(W, axis=1, keepdims=True) + 1e-8)).float()



def instala_vocab(base, E_new):
    m = copy.deepcopy(base); m.resize_token_embeddings(E_new.shape[0], mean_resizing=False)
    m.get_input_embeddings().weight.data.copy_(E_new.to(device)); m.tie_weights(); return m

## Paso 1 · TODO 1: inicialización por la media

Los tokens **compartidos** se copian (ya lo hace `base_copia`); los **nuevos** reciben la media de $E_a$. Es la línea base de `transformers`.

In [ ]:
def init_media(E_A, tkA, tkB):
    """Matriz [V_B, d]: los tokens COMPARTIDOS se copian; los NUEVOS reciben la media de todas las filas de E_A."""
    E, nuevo = base_copia(E_A, tkA, tkB)          # E: [V_B, d] con los compartidos copiados; `nuevo`: índices por inicializar
    # TODO 1: rellena E[nuevo] con la media de E_A sobre el vocabulario antiguo
    raise NotImplementedError("TODO 1")

<details>
<summary>🪜 Pista 1</summary>

`E[nuevo] = ...` con `E_A.mean(0)` (vector de dimensión d). Pytorch hace *broadcasting* sobre las filas indexadas.

</details>

## Paso 2 · TODO 2: FVT (media de la descomposición)

Para un token nuevo `t`, el tokenizer antiguo lo parte en subtokens: `descomposicion(tkA, VB[i])` devuelve sus ids. El embedding de `t` es la media de los embeddings de esos ids.

In [ ]:
def init_fvt(E_A, tkA, tkB):
    """Fast Vocabulary Transfer: cada token nuevo = media de los embeddings de su descomposición con el tokenizer ANTIGUO."""
    E, nuevo = base_copia(E_A, tkA, tkB); VB = vocab_ordenado(tkB)
    # TODO 2: para cada i en `nuevo`: ids = descomposicion(tkA, VB[i])  ->  E[i] = media de E_A[ids]
    raise NotImplementedError("TODO 2")

<details>
<summary>🪜 Pista 1</summary>

Es un bucle `for i in nuevo:`; `E_A[ids]` con `ids` una lista de enteros devuelve una matriz [len(ids), d].

</details>

<details>
<summary>🪜 Pista 2</summary>

`E[i] = E_A[descomposicion(tkA, VB[i])].mean(0)`.

</details>

## Paso 3 · TODO 3: FOCUS-lite (sparsemax sobre los tokens compartidos)

Los pesos salen de la similitud en el espacio auxiliar `aux_B` (ya normalizado): `Ac @ aux_B[i]` son cosenos entre el token nuevo `i` y cada token compartido. `sparsemax` los convierte en pesos que suman 1 con muchos ceros exactos.

In [ ]:
def init_focus(E_A, tkA, tkB, aux_B):
    """FOCUS-lite: cada token nuevo = combinación sparsemax de los tokens COMPARTIDOS.
    aux_B: [V_B, k] embeddings auxiliares NORMALIZADOS de los tokens de B (entrenados con texto del destino)."""
    E, nuevo = base_copia(E_A, tkA, tkB); vA, VB = tkA.get_vocab(), vocab_ordenado(tkB)
    comp = [i for i, t in enumerate(VB) if t in vA]            # índices (en B) de los tokens compartidos
    # TODO 3: para cada i en `nuevo`: w = sparsemax(aux_B[comp] @ aux_B[i])  ->  E[i] = suma_j w_j * E[comp[j]]
    raise NotImplementedError("TODO 3")

<details>
<summary>🪜 Pista 1</summary>

Construye `Ec = E[comp]` (embeddings de compartidos en el orden de `comp`) y `Ac = aux_B[comp]` una vez, fuera del bucle.

</details>

<details>
<summary>🪜 Pista 2</summary>

Dentro del bucle: `w = sparsemax(Ac @ aux_B[i])`; `E[i] = (w[:, None] * Ec).sum(0)`. Si `w.sum()==0` (no debería) usa la media.

</details>

## Paso 4 · TODO 4: ajuste en dos fases

Primero entrenas **solo la matriz de embeddings** (barato; la cabeza está atada), luego **todo** el modelo. `entrena_lm(model, ids, pasos, lr, params, log_cada, evalfn)` devuelve la curva `[(paso, bpb)]`.

In [ ]:
def adapta(base, E_new, ids, K1, K2, evalfn, log_cada=25):
    """Instala E_new como matriz de embeddings (instala_vocab), entrena K1 pasos SOLO embeddings (lr 3e-3) y luego K2 pasos
    con todo el modelo (lr 1e-3). Devuelve (modelo, curva) con curva = [(paso, bpb), ...] (usa entrena_lm con log_cada y evalfn)."""
    m = instala_vocab(base, E_new)
    # TODO 4a: congela todos los parámetros salvo m.get_input_embeddings().weight (la cabeza está atada) y entrena K1 pasos
    # TODO 4b: descongela todo y entrena K2 pasos; concatena las curvas (desplaza los pasos de la 2ª fase en K1)
    raise NotImplementedError("TODO 4")

<details>
<summary>🪜 Pista 1</summary>

Para congelar: `for p in m.parameters(): p.requires_grad = False` y luego `m.get_input_embeddings().weight.requires_grad = True`; pasa `params=[m.get_input_embeddings().weight]` al optimizador.

</details>

<details>
<summary>🪜 Pista 2</summary>

Fase 1: `c1 = entrena_lm(m, ids, K1, lr=3e-3, params=[W], log_cada=log_cada, evalfn=evalfn)`. Fase 2: reactiva `requires_grad` en todo y `c2 = entrena_lm(m, ids, K2, lr=1e-3, ...)`. Devuelve `c1[:-1] + [(s + K1, v) for s, v in c2]` (quita el último punto de c1: es el primero de c2).

</details>

In [ ]:
# --- Comprobaciones unitarias de tus inicializaciones (no las cambies) ---
def comprueba_inits():
    ok = {}
    E_A = BASE.get_input_embeddings().weight.detach().cpu().float(); vA = TK_A.get_vocab(); VB = vocab_ordenado(TK_B)
    comp = next(i for i, t in enumerate(VB) if t in vA and len(t) > 1); nue = next(i for i, t in enumerate(VB) if t not in vA and len(t) > 2)
    try:
        Em = init_media(E_A, TK_A, TK_B)
        ok["media: compartidos copiados"] = bool(torch.allclose(Em[comp], E_A[vA[VB[comp]]]))
        ok["media: nuevos = media de E_A"] = bool(torch.allclose(Em[nue], E_A.mean(0), atol=1e-6))
    except NotImplementedError: pass
    try:
        Ef = init_fvt(E_A, TK_A, TK_B); ids = descomposicion(TK_A, VB[nue])
        ok["FVT: nuevo = media de su descomposición"] = bool(torch.allclose(Ef[nue], E_A[ids].mean(0), atol=1e-6))
    except NotImplementedError: pass
    try:
        Eo = init_focus(E_A, TK_A, TK_B, aux_B)
        ok["FOCUS: compartidos copiados"] = bool(torch.allclose(Eo[comp], E_A[vA[VB[comp]]]))
        ok["FOCUS: nuevos dentro de la envolvente convexa (norma <= máx de los compartidos)"] = bool(Eo[nue].norm() <= E_A.norm(dim=1).max() + 1e-4)
    except NotImplementedError: pass
    for k, v in ok.items(): print("✅" if v else "❌", k)
    if not ok: print("Pendiente: implementa los TODO 1-3")
    return ok

In [ ]:
E_A = BASE.get_input_embeddings().weight.detach().cpu().float()
IDS_B_TGT = ids_stream(TK_B, TGT); IDS_A_TGT = ids_stream(TK_A, TGT)
aux_B = aux_embeddings(IDS_B_TGT, TK_B.get_vocab_size())
eval_B = lambda m: bpb(m, TK_B, VA_ES)
K1, K2 = (100, 200) if FAST_DEV_RUN else (500, 2000)
RES = {"bpb_A_sin_adaptar": bpb(BASE, TK_A, VA_ES),
       "fert": bytes_x_token(TK_B, VA_ES) / bytes_x_token(TK_A, VA_ES),
       "bpb_src_antes": bpb(BASE, TK_A, VA_SRC)}
print(f"modelo base con A sobre ES (sin adaptar): {RES['bpb_A_sin_adaptar']:.3f} bpb | fertilidad B/A = {RES['fert']:.2f}x bytes por token")
comprueba_inits()

In [ ]:
# --- Flujo (no lo cambies): corre las inicializaciones y la adaptación ---
try:
    torch.manual_seed(SEED)
    E_rand = torch.randn(TK_B.get_vocab_size(), E_A.shape[1]) * E_A.std()                    # control: todo aleatorio (sin copiar compartidos)
    m_r, c_r = adapta(BASE, E_rand, IDS_B_TGT, K1, K2, eval_B)
    RES["curva_aleatoria"] = c_r; RES["bpb0_aleatoria"] = c_r[0][1]
    RES["modelos"], RES["E"] = {}, {}
    for nombre, E_new in [("media", init_media(E_A, TK_A, TK_B)), ("FVT", init_fvt(E_A, TK_A, TK_B)), ("FOCUS-lite", init_focus(E_A, TK_A, TK_B, aux_B))]:
        m, c = adapta(BASE, E_new, IDS_B_TGT, K1, K2, eval_B)
        RES["modelos"][nombre] = m; RES["E"][nombre] = E_new; RES.setdefault("curvas", {})[nombre] = c
        print(f"{nombre:11s} bpb 0-shot={c[0][1]:.3f} | tras {K1} pasos (emb) = {[v for s, v in c if s == K1][0]:.3f} | final={c[-1][1]:.3f}")
    print(f"{'aleatoria':11s} bpb 0-shot={c_r[0][1]:.3f} | tras {K1} pasos (emb) = {[v for s, v in c_r if s == K1][0]:.3f} | final={c_r[-1][1]:.3f}")
    RES["bpbK1_aleatoria"] = [v for s, v in c_r if s == K1][0]
    RES["bpbK1_fvt"] = [v for s, v in RES["curvas"]["FVT"] if s == K1][0]
    RES["mejor"] = min(RES["curvas"], key=lambda n: RES["curvas"][n][-1][1])
    RES["bpb_final"] = RES["curvas"][RES["mejor"]][-1][1]
    RES["bpb0_fvt"] = RES["curvas"]["FVT"][0][1]
except NotImplementedError as e:
    print("Pendiente:", e)

In [ ]:
# --- Regresión: ¿cuánto olvida el modelo adaptado del inglés/código? (se mide con el tokenizer B sobre EN+código) ---
if "mejor" in RES:
    RES["bpb_src_despues"] = bpb(RES["modelos"][RES["mejor"]], TK_B, VA_SRC)
    m0 = instala_vocab(BASE, RES["E"][RES["mejor"]])           # misma init, SIN ajuste: referencia justa
    RES["bpb_src_B_sin_adaptar"] = bpb(m0, TK_B, VA_SRC)
    print(f"EN+código con tokenizer B: sin ajustar {RES['bpb_src_B_sin_adaptar']:.3f} -> ajustado al ES {RES['bpb_src_despues']:.3f}")
else:
    print("Completa los TODO para ver la regresión.")

In [ ]:
if "curvas" in RES:
    fig, ax = plt.subplots(figsize=(8, 4))
    for n, c in RES["curvas"].items(): ax.plot([s for s, _ in c], [v for _, v in c], lw=2, label=n)
    ax.plot([s for s, _ in RES["curva_aleatoria"]], [v for _, v in RES["curva_aleatoria"]], "k--", label="aleatoria")
    ax.axhline(RES["bpb_A_sin_adaptar"], color="crimson", ls=":", label="base con tokenizer A (sin adaptar)")
    ax.axvline(K1, color="gray", lw=.6, ls=":"); ax.set_ylim(top=min(ax.get_ylim()[1], 6)); ax.set_xlabel("pasos"); ax.set_ylabel("bpb en ES (val)")
    ax.set_title("Adaptación corta tras cambiar de tokenizer"); ax.legend(fontsize=8); plt.show()
else:
    print("Completa los TODO para ver las curvas.")

> **👀 Qué debes observar**
>
> En el paso 0 las inits apenas difieren en este banco diminuto, pero al llegar a la línea vertical (fin de la fase 1, solo embeddings) FVT debe quedar claramente por debajo de la aleatoria (la referencia mide ≈ 0.70 frente a ≈ 0.78 bpb): esa distancia es lo que ahorra una buena inicialización. Después de la línea (fase 2, todo el modelo) las curvas convergen: con más pasos la init importa menos. La línea roja es el modelo original con su tokenizer A; si alguna curva la cruza hacia abajo, el tokenizer nuevo ya **modela mejor** el español por byte.

## Paso 5 · Conclusión y rúbrica

Escribe tu recomendación para el CTO: ¿cambiar el tokenizer compensa? ¿qué init usarías? ¿qué **no** puedes concluir con este banco (una semilla, modelo diminuto, datos sintéticos, ~300 pasos)?

In [ ]:
CONCLUSION = ""   # TODO 5: tu nota técnica (>= 300 caracteres)

In [ ]:
THR = {'fert_min': 1.6, 'mejora_emb': 0.03, 'bpb_final_max': 0.3, 'regresion_max': 0.6}

def rubrica(RES, th=THR):
    filas = []
    chk = lambda n, ok, d: filas.append((n, ok, d))
    ok_u = comprueba_inits()
    chk("R1 tus inicializaciones pasan las comprobaciones unitarias (media, FVT, FOCUS)", len(ok_u) >= 5 and all(ok_u.values()), f"{sum(ok_u.values())}/{len(ok_u)}")
    chk("R2 el tokenizer nuevo reduce los tokens en ES (fertilidad B/A >= %.1fx)" % th["fert_min"], RES.get("fert", 0) >= th["fert_min"], f"{RES.get('fert')}")
    chk("R3 FVT mejora a la init aleatoria tras la fase de embeddings (bpb FVT <= aleatoria - %.2f)" % th["mejora_emb"],
        RES.get("bpbK1_fvt") is not None and RES["bpbK1_fvt"] <= RES["bpbK1_aleatoria"] - th["mejora_emb"], f"FVT={RES.get('bpbK1_fvt')} vs aleatoria={RES.get('bpbK1_aleatoria')}")
    chk("R4 tras la adaptación, bpb en ES <= %.2f con la mejor init" % th["bpb_final_max"], RES.get("bpb_final", 99) <= th["bpb_final_max"], f"{RES.get('bpb_final')} ({RES.get('mejor')})")
    chk("R5 la adaptación mejora al modelo base sin adaptar con A (menos bpb en ES)", RES.get("bpb_final", 99) < RES.get("bpb_A_sin_adaptar", 0), f"{RES.get('bpb_final')} < {RES.get('bpb_A_sin_adaptar')}")
    reg = None if "bpb_src_despues" not in RES else RES["bpb_src_despues"] / RES["bpb_src_B_sin_adaptar"] - 1
    chk("R6 regresión en EN+código (aumento relativo de bpb) <= %d %%" % (th["regresion_max"] * 100), reg is not None and reg <= th["regresion_max"], f"{reg}")
    chk("R7 conclusión escrita (>= 300 caracteres) en CONCLUSION", len(globals().get("CONCLUSION", "")) >= 300, f"{len(globals().get('CONCLUSION', ''))} caracteres")
    for n, ok, d in filas: print(("✅" if ok else "❌"), n, "|", d)
    print(f"\n{sum(o for _, o, _ in filas)}/{len(filas)} criterios")

rubrica(RES)

---
## ⛔ SPOILER — intenta resolverlo primero

Debajo está la solución de referencia **completa y ejecutable**: redefine las funciones y vuelve a correr el flujo.

### Solución de referencia

In [ ]:
def init_media(E_A, tkA, tkB):
    E, nuevo = base_copia(E_A, tkA, tkB)
    E[nuevo] = E_A.mean(0); return E

In [ ]:
def init_fvt(E_A, tkA, tkB):
    E, nuevo = base_copia(E_A, tkA, tkB); VB = vocab_ordenado(tkB)
    for i in nuevo:
        E[i] = E_A[descomposicion(tkA, VB[i])].mean(0)
    return E

In [ ]:
def init_focus(E_A, tkA, tkB, aux_B):
    E, nuevo = base_copia(E_A, tkA, tkB); vA, VB = tkA.get_vocab(), vocab_ordenado(tkB)
    comp = [i for i, t in enumerate(VB) if t in vA]; Ec = E[comp]; Ac = aux_B[comp]
    for i in nuevo:
        w = sparsemax(Ac @ aux_B[i])
        E[i] = (w[:, None] * Ec).sum(0) if w.sum() > 0 else E_A.mean(0)
    return E

In [ ]:
def adapta(base, E_new, ids, K1, K2, evalfn, log_cada=25):
    m = instala_vocab(base, E_new)
    for p_ in m.parameters(): p_.requires_grad = False
    m.get_input_embeddings().weight.requires_grad = True
    c1 = entrena_lm(m, ids, K1, lr=3e-3, params=[m.get_input_embeddings().weight], log_cada=log_cada, evalfn=evalfn)
    for p_ in m.parameters(): p_.requires_grad = True
    c2 = entrena_lm(m, ids, K2, lr=1e-3, log_cada=log_cada, evalfn=evalfn)
    return m, c1[:-1] + [(s + K1, v) for s, v in c2]

In [ ]:
E_A = BASE.get_input_embeddings().weight.detach().cpu().float()
IDS_B_TGT = ids_stream(TK_B, TGT); IDS_A_TGT = ids_stream(TK_A, TGT)
aux_B = aux_embeddings(IDS_B_TGT, TK_B.get_vocab_size())
eval_B = lambda m: bpb(m, TK_B, VA_ES)
K1, K2 = (100, 200) if FAST_DEV_RUN else (500, 2000)
RES = {"bpb_A_sin_adaptar": bpb(BASE, TK_A, VA_ES),
       "fert": bytes_x_token(TK_B, VA_ES) / bytes_x_token(TK_A, VA_ES),
       "bpb_src_antes": bpb(BASE, TK_A, VA_SRC)}
print(f"modelo base con A sobre ES (sin adaptar): {RES['bpb_A_sin_adaptar']:.3f} bpb | fertilidad B/A = {RES['fert']:.2f}x bytes por token")
comprueba_inits()

In [ ]:
# --- Flujo (no lo cambies): corre las inicializaciones y la adaptación ---
try:
    torch.manual_seed(SEED)
    E_rand = torch.randn(TK_B.get_vocab_size(), E_A.shape[1]) * E_A.std()                    # control: todo aleatorio (sin copiar compartidos)
    m_r, c_r = adapta(BASE, E_rand, IDS_B_TGT, K1, K2, eval_B)
    RES["curva_aleatoria"] = c_r; RES["bpb0_aleatoria"] = c_r[0][1]
    RES["modelos"], RES["E"] = {}, {}
    for nombre, E_new in [("media", init_media(E_A, TK_A, TK_B)), ("FVT", init_fvt(E_A, TK_A, TK_B)), ("FOCUS-lite", init_focus(E_A, TK_A, TK_B, aux_B))]:
        m, c = adapta(BASE, E_new, IDS_B_TGT, K1, K2, eval_B)
        RES["modelos"][nombre] = m; RES["E"][nombre] = E_new; RES.setdefault("curvas", {})[nombre] = c
        print(f"{nombre:11s} bpb 0-shot={c[0][1]:.3f} | tras {K1} pasos (emb) = {[v for s, v in c if s == K1][0]:.3f} | final={c[-1][1]:.3f}")
    print(f"{'aleatoria':11s} bpb 0-shot={c_r[0][1]:.3f} | tras {K1} pasos (emb) = {[v for s, v in c_r if s == K1][0]:.3f} | final={c_r[-1][1]:.3f}")
    RES["bpbK1_aleatoria"] = [v for s, v in c_r if s == K1][0]
    RES["bpbK1_fvt"] = [v for s, v in RES["curvas"]["FVT"] if s == K1][0]
    RES["mejor"] = min(RES["curvas"], key=lambda n: RES["curvas"][n][-1][1])
    RES["bpb_final"] = RES["curvas"][RES["mejor"]][-1][1]
    RES["bpb0_fvt"] = RES["curvas"]["FVT"][0][1]
except NotImplementedError as e:
    print("Pendiente:", e)


# --- Regresión: ¿cuánto olvida el modelo adaptado del inglés/código? (se mide con el tokenizer B sobre EN+código) ---
if "mejor" in RES:
    RES["bpb_src_despues"] = bpb(RES["modelos"][RES["mejor"]], TK_B, VA_SRC)
    m0 = instala_vocab(BASE, RES["E"][RES["mejor"]])           # misma init, SIN ajuste: referencia justa
    RES["bpb_src_B_sin_adaptar"] = bpb(m0, TK_B, VA_SRC)
    print(f"EN+código con tokenizer B: sin ajustar {RES['bpb_src_B_sin_adaptar']:.3f} -> ajustado al ES {RES['bpb_src_despues']:.3f}")
else:
    print("Completa los TODO para ver la regresión.")

In [ ]:
if "curvas" in RES:
    fig, ax = plt.subplots(figsize=(8, 4))
    for n, c in RES["curvas"].items(): ax.plot([s for s, _ in c], [v for _, v in c], lw=2, label=n)
    ax.plot([s for s, _ in RES["curva_aleatoria"]], [v for _, v in RES["curva_aleatoria"]], "k--", label="aleatoria")
    ax.axhline(RES["bpb_A_sin_adaptar"], color="crimson", ls=":", label="base con tokenizer A (sin adaptar)")
    ax.axvline(K1, color="gray", lw=.6, ls=":"); ax.set_ylim(top=min(ax.get_ylim()[1], 6)); ax.set_xlabel("pasos"); ax.set_ylabel("bpb en ES (val)")
    ax.set_title("Adaptación corta tras cambiar de tokenizer"); ax.legend(fontsize=8); plt.show()
else:
    print("Completa los TODO para ver las curvas.")

In [ ]:
CONCLUSION = (
    f"Recomendación: cambiar el tokenizer compensa si el español es tráfico relevante: los tokens por byte bajan a ~{1/RES['fert']:.2f}x (cada byte necesita menos pasos del transformer). "
    f"Mejor init en nuestro banco: {RES['mejor']}, con bpb en ES de {RES['bpb_final']:.3f} tras el ajuste frente a {RES['bpb_A_sin_adaptar']:.3f} del modelo base sin adaptar (tokenizer A). "
    f"Cero-shot, FVT = {RES['bpb0_fvt']:.3f} vs aleatoria = {RES['bpb0_aleatoria']:.3f} bpb. La regresión en inglés/código con el tokenizer nuevo fue de {RES['bpb_src_despues']/RES['bpb_src_B_sin_adaptar']-1:+.1%}. "
    "Límites: modelo diminuto preentrenado por nosotros, corpus pequeño/sintético, una semilla y ~300 pasos; no sabemos si las diferencias entre inits se mantienen con GPT-2 real ni con datos de Wikipedia. "
    "Siguiente paso: repetir en modo completo (L4) con GPT-2 small, varias semillas y una evaluación de tarea real (no solo bpb)."
)
rubrica(RES)

## Retos extra (nivel experto)

1. Implementa **WECHSEL-lite** (Procrustes entre espacios auxiliares + softmax con τ=0.1 y k=10) y compáralo con FOCUS-lite.
2. Mide cuántos pasos de ajuste necesita cada init para **igualar** al control «seguir con A» (curva de bpb vs pasos): ¿cuál «ahorra» más?
3. Añade **ruido de la covarianza** (Hewitt) a la init por media y mide si acelera la separación de los tokens nuevos.
4. Prueba un vocabulario B **más grande** (p. ej. 1536) y otro **más pequeño**: ¿cómo cambian fertilidad, bpb cero-shot y bpb final?
5. En modo completo (L4): usa **GPT-2 small** real con tokenizer B de 24 000 tokens entrenado en ES+código; compara con `resize_token_embeddings(mean_resizing=True)` de Hugging Face.
6. **ZeTT-mini:** entrena una MLP que prediga el embedding de un token a partir de la media de su descomposición y úsala como init.

## Reflexión

* ¿Qué parte del beneficio viene de **tokens más largos** (eficiencia) y qué parte de una mejor **calidad** del modelo? ¿Cómo lo separarías?
* ¿Qué haría falta para extender esto a un **embedder** de Políglota (encoder + pooling)? ¿Qué cambia si no hay *weight tying*?
* ¿Cuántos tokens de ajuste pagarías para cerrar el hueco? (ZeTT reporta < 1B tokens para decodificadores de 7B.)
* ¿Cómo comunicarías al CTO que *«la pérdida bajó»* no es una prueba de mejora tras un cambio de tokenizer?